In [1]:
# 이번 실험의 목적
# 이번 fine-tuning
# 모델 = ResNet18
# 시작 가중치 = Baseline 최고 성능 모델
# 학습 계층 = Layer4 + FC
# LR = 0.0001
# Epoch = 추가 5
# Loss = BCEWithLogitsLoss
# Threshold = 0.5 / 0.3

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

DATA_DIR = Path("../data")
IMAGE_DIR = DATA_DIR / "train_images_224"

train_data = pd.read_csv(DATA_DIR / "train_split.csv")
valid_data = pd.read_csv(DATA_DIR / "valid_split.csv")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Train:", len(train_data))
print("Valid:", len(valid_data))

Device: cuda
Train: 14906
Valid: 3726


In [3]:
classes = [
    "scab",
    "healthy",
    "frog_eye_leaf_spot",
    "complex",
    "rust",
    "powdery_mildew"
]


def multi_hot_encode(labels):
    label_list = labels.split()

    encoded = []

    for cls in classes:
        if cls in label_list:
            encoded.append(1)
        else:
            encoded.append(0)

    return encoded

class PlantDataset(Dataset):
    def __init__(self, df, image_dir, transforms = None):
        self.df = df.reset_index(drop = True)
        self.image_dir = Path(image_dir)
        self.transform = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["image"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            multi_hot_encode(row["labels"]),
            dtype = torch.float32
        )

        return image, label

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

train_dataset = PlantDataset(
    train_data, IMAGE_DIR, transform
)

valid_dataset = PlantDataset(
    valid_data, IMAGE_DIR, transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers=0
)

In [5]:
model = models.resnet18(weights=None)   # 이미 학습한 모델의 가중치를 쓸거기 때문에 weights = None

model.fc = nn.Linear(
    model.fc.in_features,
    6
)

model.load_state_dict(
    torch.load(
        "best_resnet18.pth",
        map_location='cpu',
        weights_only=True
    )
)

model = model.to(device)

In [6]:
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

In [7]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [8]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr = 0.0001,
    weight_decay=0.01
)

In [9]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    # BatchNorm Running Statistics 고정
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0

    for images, labels in tqdm(loader, desc = "Training"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)


    return total_loss / len(loader.dataset)

c:\Users\SSAFY\Desktop\ParkJaeWon\AI-practice\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)
            loss = criterion(output, labels)

            probs = torch.sigmoid(output)

            total_loss += loss.item() * images.size(0)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())


    valid_loss = total_loss / len(loader.dataset)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)


    preds_05 = (all_probs >= 0.5).astype(int)
    preds_03 = (all_probs >= 0.3).astype(int)

    f1_05 = f1_score(
        all_labels, preds_05,
        average="macro",
        zero_division=0
    )

    f1_03 = f1_score(
        all_labels, preds_03,
        average="macro",
        zero_division=0
    )

    return valid_loss, f1_05, f1_03



In [11]:
EPOCHS = 5

train_losses = []
valid_losses = []
f1_05_scores = []
f1_03_scores = []

best_f1 = -1

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    valid_loss, f1_05, f1_03 = validate(
        model, valid_loader, criterion, device
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    f1_05_scores.append(f1_05)
    f1_03_scores.append(f1_03)

    print(
        f"Epoch [{epoch+1} / {EPOCHS}]"
        f"Train Loss: {train_loss:.4f}"
        f"Valid Loss: {valid_loss:.4f}"
        f"F1(0.5): {f1_05:.4f}"
        f"F1(0.3): {f1_03:.4f}"
    )

    if f1_05 > best_f1:
        best_f1 = f1_05

        torch.save(
            model.state_dict(),
            "best_resnet18_finetuned.pth"
        )


print(f"\nBest Valid Macro F1 (0.5): {best_f1:.4f}")

Training: 100%|██████████| 466/466 [00:27<00:00, 16.93it/s]


Epoch [1 / 5]Train Loss: 0.2100Valid Loss: 0.2117F1(0.5): 0.7372F1(0.3): 0.7685


Training: 100%|██████████| 466/466 [00:26<00:00, 17.30it/s]


Epoch [2 / 5]Train Loss: 0.1422Valid Loss: 0.1540F1(0.5): 0.8023F1(0.3): 0.8230


Training: 100%|██████████| 466/466 [00:26<00:00, 17.45it/s]


Epoch [3 / 5]Train Loss: 0.0992Valid Loss: 0.1944F1(0.5): 0.7988F1(0.3): 0.8100


Training: 100%|██████████| 466/466 [00:26<00:00, 17.84it/s]


Epoch [4 / 5]Train Loss: 0.0651Valid Loss: 0.1623F1(0.5): 0.8278F1(0.3): 0.8279


Training: 100%|██████████| 466/466 [00:26<00:00, 17.78it/s]


Epoch [5 / 5]Train Loss: 0.0415Valid Loss: 0.2057F1(0.5): 0.8038F1(0.3): 0.8107

Best Valid Macro F1 (0.5): 0.8278


In [12]:
# Train Loss와 Validation Loss가 모두 감소하고 있음. 현재 학습곡선만 보면 전형적인 Overfitting 패턴은 나타나지 않음

# Threshold 0.5가 0.3보다 좋아졌어. Fine-tuning 후 모델의 예측 확률 분포가 달라졌다는 의미
# 이전에 선택한 Threshold가 새로운 모델에서도 최적이라는 보장은 없음

# Validation 성능이 매우 높아졌어. 실제로 모델이 질병의 특징을 훨씬 잘 학습했을 수도 있음
# 데이터 누수나 평가 코드 오류도 확인

In [14]:
print("Train Dataset:", type(train_loader.dataset))
print("Valid_Dataset:", type(valid_loader.dataset))

print("Train:", len(train_loader.dataset))
print("Valid:", len(valid_loader.dataset))

train_images = set(train_loader.dataset.df["image"])
valid_images = set(valid_loader.dataset.df["image"])

print("겹치는 이미지:", len(train_images & valid_images))

print("Train 경로:", train_loader.dataset.image_dir)
print("Valid 경로:", valid_loader.dataset.image_dir)

Train Dataset: <class '__main__.PlantDataset'>
Valid_Dataset: <class '__main__.PlantDataset'>
Train: 14906
Valid: 3726
겹치는 이미지: 0
Train 경로: ..\data\train_images_224
Valid 경로: ..\data\train_images_224


In [16]:

from sklearn.metrics import classification_report

# 클래스 별 성능

model.load_state_dict(
    torch.load(
        "best_resnet18_finetuned.pth",
        map_location=device,
        weights_only=True
    )
)

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.sigmoid(outputs)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

preds = (all_probs >= 0.5).astype(int)

print(classification_report(
    all_labels,
    preds,
    target_names=classes,
    zero_division=0
))

                    precision    recall  f1-score   support

              scab       0.92      0.75      0.83      1123
           healthy       0.94      0.93      0.93       939
frog_eye_leaf_spot       0.82      0.77      0.80       893
           complex       0.71      0.58      0.64       432
              rust       0.92      0.87      0.89       407
    powdery_mildew       0.83      0.94      0.88       252

         micro avg       0.87      0.80      0.84      4046
         macro avg       0.86      0.81      0.83      4046
      weighted avg       0.87      0.80      0.83      4046
       samples avg       0.84      0.83      0.83      4046



In [17]:
model = models.resnet18(weights = None)
model.fc = nn.Linear(model.fc.in_features, 6)

model.load_state_dict(
    torch.load(
        "best_resnet18.pth",
        map_location = "cpu",
        weights_only=True
    )
)

model = model.to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr = 0.0001,
    weight_decay=0.01
)

In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    # BatchNorm Running Statistics 고정
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0

    for images, labels in tqdm(loader, desc = "Training"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)


    return total_loss / len(loader.dataset)

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            output = model(images)
            loss = criterion(output, labels)

            probs = torch.sigmoid(output)

            total_loss += loss.item() * images.size(0)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())


    valid_loss = total_loss / len(loader.dataset)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)


    preds_05 = (all_probs >= 0.5).astype(int)
    preds_03 = (all_probs >= 0.3).astype(int)

    f1_05 = f1_score(
        all_labels, preds_05,
        average="macro",
        zero_division=0
    )

    f1_03 = f1_score(
        all_labels, preds_03,
        average="macro",
        zero_division=0
    )

    return valid_loss, f1_05, f1_03



In [18]:
EPOCHS = 5

train_losses = []
valid_losses = []
f1_05_scores = []
f1_03_scores = []

best_f1 = -1

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    valid_loss, f1_05, f1_03 = validate(
        model, valid_loader, criterion, device
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    f1_05_scores.append(f1_05)
    f1_03_scores.append(f1_03)

    print(
        f"Epoch [{epoch+1} / {EPOCHS}]"
        f"Train Loss: {train_loss:.4f}"
        f"Valid Loss: {valid_loss:.4f}"
        f"F1(0.5): {f1_05:.4f}"
        f"F1(0.3): {f1_03:.4f}"
    )

    if f1_05 > best_f1:
        best_f1 = f1_05

        torch.save(
            model.state_dict(),
            "best_resnet18_fc_only.pth"
        )


print(f"\nBest Valid Macro F1 (0.5): {best_f1:.4f}")

Training: 100%|██████████| 466/466 [00:24<00:00, 18.97it/s]


Epoch [1 / 5]Train Loss: 0.2456Valid Loss: 0.2506F1(0.5): 0.6541F1(0.3): 0.7027


Training: 100%|██████████| 466/466 [00:24<00:00, 19.21it/s]


Epoch [2 / 5]Train Loss: 0.2452Valid Loss: 0.2506F1(0.5): 0.6567F1(0.3): 0.7009


Training: 100%|██████████| 466/466 [00:23<00:00, 19.61it/s]


Epoch [3 / 5]Train Loss: 0.2447Valid Loss: 0.2500F1(0.5): 0.6607F1(0.3): 0.7011


Training: 100%|██████████| 466/466 [00:24<00:00, 19.06it/s]


Epoch [4 / 5]Train Loss: 0.2443Valid Loss: 0.2499F1(0.5): 0.6562F1(0.3): 0.7029


Training: 100%|██████████| 466/466 [00:25<00:00, 18.63it/s]


Epoch [5 / 5]Train Loss: 0.2439Valid Loss: 0.2493F1(0.5): 0.6617F1(0.3): 0.7009

Best Valid Macro F1 (0.5): 0.6617


In [23]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "leo10365/plant_pathology_2021_resnet18"

checkpoints = {
    "best_resnet18.pth":
    "checkpoints/baseline/best_resnet18.pth",

    "best_resnet18_finetuned.pth":
    "checkpoints/finetuning/best_resnet18_finetuned.pth"
}

for local_path, hub_path in checkpoints.items():
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=hub_path,
        repo_id=repo_id
    )

Processing Files (1 / 1): 100%|██████████| 44.8MB / 44.8MB,   ???B/s  












Processing Files (1 / 1): 100%|██████████| 44.8MB / 44.8MB, 4.01MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Validating: ██████████| 100%            
Processing Files (1 / 1): 100%|██████████| 44.8MB / 44.8MB, 3.14MB/s  






Processing Files (1 / 1): 100%|██████████| 44.8MB / 44.8MB, 2.99MB/s  
New Data Upload: 100%|██████████| 34.8MB / 34.8MB, 2.44MB/s  
Validating: ██████████| 100%            
